# MCP Gateway Agent Demo

Same agent, same code. Only the **URL** changes.

**The only change between "Managed" and "Clone" is the MCP endpoint URL.**

| Scenario | URL | Result |
|----------|-----|--------|
| Managed MCP | `WORKSPACE_HOST/api/2.0/mcp/genie/{space_id}` | Direct to Genie API |
| Clone MCP (1st) | `APP_HOST/api/2.0/mcp/genie/{gateway_id}` | Via Gateway (queue + retry) |
| Clone MCP (2nd) | `APP_HOST/api/2.0/mcp/genie/{gateway_id}` | Instant (semantic cache) |

In [ ]:
%pip install 'openai-agents[mcp]' -q

In [ ]:
dbutils.library.restartPython()

In [ ]:
dbutils.widgets.text("app_url", "https://genie-cache-queue-7474650836156271.aws.databricksapps.com", "App URL")
dbutils.widgets.text("space_id", "", "Genie Space ID")
dbutils.widgets.text("gateway_id", "", "Gateway ID")
dbutils.widgets.text("model", "databricks-claude-sonnet-4", "Model Serving Endpoint")

In [ ]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
WORKSPACE_HOST = w.config.host.rstrip("/")
TOKEN          = dbutils.secrets.get(scope="genie-cache", key="oauth_token")
APP_HOST       = dbutils.widgets.get("app_url").rstrip("/")
GENIE_SPACE_ID = dbutils.widgets.get("space_id")
GATEWAY_ID     = dbutils.widgets.get("gateway_id")
MODEL          = dbutils.widgets.get("model")

MANAGED_URL = f"{WORKSPACE_HOST}/api/2.0/mcp/genie/{GENIE_SPACE_ID}"
CLONE_URL   = f"{APP_HOST}/api/2.0/mcp/genie/{GATEWAY_ID}"

print(f"Managed: {MANAGED_URL}")
print(f"Clone:   {CLONE_URL}")

In [ ]:
import asyncio, time, logging
from openai import AsyncOpenAI
from agents import Agent, Runner
from agents.models.openai_chatcompletions import OpenAIChatCompletionsModel
from agents.mcp import MCPServerStreamableHttp

logging.getLogger("openai.agents").setLevel(logging.ERROR)

oai = AsyncOpenAI(base_url=f"{WORKSPACE_HOST}/serving-endpoints", api_key=TOKEN)
model = OpenAIChatCompletionsModel(model=MODEL, openai_client=oai)

AUTH = {"Authorization": f"Bearer {TOKEN}"}

async def ask(url, question):
    async with MCPServerStreamableHttp(params={"url": url, "headers": AUTH}, client_session_timeout_seconds=120) as mcp:
        agent = Agent(name="analyst", model=model, mcp_servers=[mcp])
        t0 = time.time()
        result = await Runner.run(agent, question, max_turns=30)
        elapsed = round(time.time() - t0, 1)
        print(f"  [{elapsed}s] {result.final_output[:200]}")
        return result, elapsed

print("Ready.")

## Scenario A: Managed Databricks MCP

Direct to the workspace Genie MCP endpoint — no caching, no queue.

```
ask(url=MANAGED_URL, question=...)  # workspace.cloud.databricks.com
```

In [ ]:
Q = "What are the bottom 3 nations by total revenue?"

print(f"URL: {MANAGED_URL}")
result_a, time_a = await ask(MANAGED_URL, Q)

## Scenario B: Clone MCP — first call

Same agent, same question. Only the URL points to the Gateway — queue, retry, and cache.

```
ask(url=CLONE_URL, question=...)  # app.aws.databricksapps.com
```

In [ ]:
print(f"URL: {CLONE_URL}")
result_b, time_b = await ask(CLONE_URL, Q)

## Scenario C: Clone MCP — second call (cache hit)

Same query again, same URL. Should hit the **semantic cache** — instant response.

```
ask(url=CLONE_URL, question=...)  # same call, cached
```

In [ ]:
print(f"URL: {CLONE_URL}")
result_c, time_c = await ask(CLONE_URL, Q)

## Comparison

In [ ]:
print(f"A - Managed:     {time_a}s")
print(f"B - Clone 1st:   {time_b}s")
print(f"C - Clone 2nd:   {time_c}s  (cache hit)")
print()
print(f"Same agent. Only the URL changed.")